# Tidy the thesis project

Moves every file the current pipeline (Notebooks 01–06 v2 and 08) does **not** use into one archive folder, `_archive_2026-10/`, keeping its original path inside it. **Nothing is deleted**, and the last cell undoes the move.

**How to use**
1. Run all cells with `APPLY = False` (the default). Nothing moves. You get the plan: what stays, what is archived, and what is left alone for you to decide (`review`). The plan is also saved as `tidy_plan.csv`.
2. Read the plan. If a file you still need is marked `archive`, add its name to `KEEP_EXTRA` and re-run.
3. Set `APPLY = True` and run again. The moves are logged in `_archive_2026-10/ARCHIVE_INDEX.csv`.

**What stays:** `raw/`, `processed/core|network|features/`, the v2 output folders (`outputs/tables|figures/` → `eda`, `interpretation`, `success`, `narrative_dynamics`, `corpus_audit`, `network_diagnostics`, `robustness`), the top-level tables the v2 notebooks still write under their v1 names, the Notebook 08 bricks (`fig_*`, `brick_NN_notes.txt`), the eight pipeline notebooks and this README.

**What is archived:** v1 notebooks (`notebooks/`, `embeddings/`), old data tables at the top of `processed/`, v1 figures and tables, v1 brick outputs, backups and stale label files.

**What is left alone (`review`):** anything in a folder this notebook does not know, so personal files (drafts, papers, notes) are never touched.

In [ ]:
APPLY = False                       # False = dry run (nothing moves). True = move the 'archive' files.
PROJECT_DIR = "/content/drive/MyDrive/thesis_movie_project"
ARCHIVE_NAME = "_archive_2026-10"
KEEP_EXTRA = []                     # file names (not paths) you want to keep anyway, e.g. ["my_notes.csv"]

In [ ]:
import os, re, shutil, time
from pathlib import Path
import pandas as pd

try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path(os.environ.get("THESIS_PROJECT_DIR", PROJECT_DIR))
if not ROOT.exists():
    raise FileNotFoundError(f"{ROOT} not found — check that Google Drive is mounted and PROJECT_DIR is right.")
ARCHIVE = ROOT / ARCHIVE_NAME

PIPELINE_NOTEBOOKS = {
    "01_build_curated_thesis_corpus_v2.ipynb", "02_eda_semantic_analysis_v2.ipynb", "03_sbert_network_main_v2.ipynb",
    "04_network_interpretation_v2.ipynb", "05_success_and_star_power_v2.ipynb", "06_narrative_dynamics.ipynb",
    "07_robustness_checks.ipynb", "08_thesis_bricks.ipynb", "tidy_project.ipynb",
}
ROOT_FILES_KEPT = {"README.md", ".gitattributes", ".gitignore", "requirements.txt", "tidy_plan.csv"}
V2_OUTPUT_FOLDERS = {"eda", "interpretation", "success", "narrative_dynamics", "corpus_audit", "network_diagnostics"}
BRICKS = {"01_data_construction", "02_nlp_similarity", "03_network_construction", "04_narrative_communities",
          "05_central_story_positions", "06_bridging_story_positions", "07_success_dimensions", "08_structure_success"}
# Every file name Notebooks 01–06 v2 and 08 read or write (taken from their code and saved outputs).
USED = set("""
01_integrity_report.csv 02_full_funnel.csv 02_full_funnel.png 03_representativeness_decade.csv 03_representativeness_genre.csv
03_representativeness_genre.png 03_representativeness_language.csv 04_duplicate_overviews.csv 04_overviews_sharing_title_names.csv
04_text_quality_summary.csv 05_overview_length_by_year.csv 05_overview_length_by_year.png 06_rating_spread_by_votes.csv
06_rating_spread_by_votes.png 06_threshold_sensitivity.csv 07_revenue_availability_by_visibility.csv
07_revenue_availability_by_visibility.png 07_revenue_missingness_profile.csv 07_success_signal_availability.csv
08_success_signal_correlations.csv 08_success_signal_correlations.png 08_success_signal_pair_counts.csv
09_revenue_percentile_definitions.csv betweenness_exact_vs_sampled.csv boundary_films_low_confidence.csv
bridging_films_betweenness.csv change_vs_previous_network.csv collinearity_vif.csv communities_vs_genre_nmi.csv
community_key_terms.csv community_label_carryover.csv community_label_helper_v2_manual.csv community_labels_draft_v2.csv
community_labels_v2_manual.csv community_manual_labeling_template.csv community_manual_labeling_template_labeled.csv
community_neighbourhood_long.csv community_outcome_effect_sizes.csv community_outcome_summary_descriptive.csv
community_representative_films_long.csv community_size_overview.csv community_success_summary.csv community_trends.csv
consensus_communities.csv corpus_diagnostics.csv dedup_log.csv eda_summary_notes.md edge_diagnostics_summary.csv
edges_sharing_character_names.csv event_filter_audit.csv event_filter_borderline.csv examples_foresight_crowding.csv
fig_01_correlations.csv fig_01_correlations.png fig_01_genre_composition.csv fig_01_genre_composition.png
fig_01_narrative_trends.csv fig_01_narrative_trends.png fig_02_binned_crowding_foresight.csv fig_02_binned_crowding_foresight.png
fig_02_community_neighbourhood_map.csv fig_02_community_neighbourhood_map.png fig_02_r2_by_block.csv fig_02_r2_by_block.png
fig_03_community_size_robustness.csv fig_03_community_size_robustness.png fig_03_novelty_sweet_spot.csv
fig_03_novelty_sweet_spot.png fig_03_timing_coefficients.csv fig_03_timing_coefficients.png fig_04_bridges_participation.csv
fig_04_bridges_participation.png fig_04_community_effects.csv fig_04_community_effects.png fig_04_morans_i.csv
fig_04_morans_i.png fig_05_coefficients.csv fig_05_coefficients.png fig_05_prediction_gain.csv fig_05_prediction_gain.png
fig_05_reception_by_community.csv fig_05_reception_by_community.png filtering_funnel.csv franchise_concentration_diagnostic.csv
franchise_or_name_driven_communities.csv integrity_checks.csv major_community_thesis_table.csv minor_communities.csv
model_fit_summary.csv modeling_sample_audit.csv movies_per_year_thesis_corpus.csv narrative_dynamics_notes.txt
near_duplicate_edges.csv network_audit.csv network_interpretation_basic_checks.csv network_interpretation_notes.txt
network_success_correlations.csv novelty_shape.csv out_of_sample_prediction.csv peripheral_films_candidates.csv
primary_genre_counts.csv regression_key_coefficients_summary.csv regression_results_log_revenue.csv
regression_results_popularity.csv regression_results_revenue_percentile.csv regression_results_vote_average.csv
remaining_event_like_titles_check.csv revenue_selection_profile.csv robustness_checks.csv safety_guard_report.csv
shared_title_name_edges.csv success_star_power_notes.txt timing_heterogeneity.csv timing_regressions.csv
top_20_communities_by_size.png top_betweenness_franchise_capped.csv top_betweenness_visible_franchise_capped.csv
top_pagerank_franchise_capped.csv top_pagerank_visible_franchise_capped.csv top_visible_central_films_per_community.csv
top_weighted_degree_franchise_capped.csv top_weighted_degree_raw.csv top_weighted_degree_visible_franchise_capped.csv
v1_collinearity_demo.csv
""".split()) | set(KEEP_EXTRA)

def classify(rel: Path):
    """Return (action, reason) for a path relative to the project root."""
    p, name = rel.parts, rel.name
    if name in KEEP_EXTRA:
        return "keep", "listed in KEEP_EXTRA"
    top = p[0]
    if top in (".git", ARCHIVE_NAME) or top.startswith("_archive_"):
        return "skip", "already archived / git"
    if len(p) == 1:                                            # files in the project root
        base = re.sub(r" \(\d+\)(?=\.ipynb$)", "", name)       # Drive upload copies: "08_thesis_bricks (1).ipynb"
        if base in PIPELINE_NOTEBOOKS or name in ROOT_FILES_KEPT:
            return "keep", "pipeline notebook / project file"
        if name.endswith(".ipynb"):
            return "archive", "old or duplicate notebook"
        return "review", "unknown file in the project root"
    if top == "raw":
        return "keep", "raw data"
    if top in ("notebooks", "embeddings"):
        return "archive", "v1 notebooks"
    if top == "tools":
        return "keep", "tools"
    if top == "processed":
        if len(p) == 2:
            return "archive", "old v1/v3 data table (v2 uses processed/core|network|features)"
        if p[1] in ("core", "network", "features"):
            if name == "df_revenue_analysis_enriched.csv":
                return "archive", "no longer produced (Notebook 02 v2)"
            return "keep", "v2 data"
        return "review", "unknown folder in processed/"
    if top == "outputs":
        if len(p) < 3:
            return "review", "unknown file in outputs/"
        kind = p[1]
        if kind in ("tables", "figures"):
            if len(p) == 3:
                return ("keep", "written by a v2 notebook") if name in USED else ("archive", "v1 output")
            if p[2] == "robustness":
                return "keep", "written by Notebook 07"
            if p[2] in V2_OUTPUT_FOLDERS:
                return ("keep", "written by a v2 notebook") if name in USED else ("archive", "file from an older version")
            return "archive", "v1 output folder"
        if kind == "thesis_bricks":
            if len(p) == 3:
                return ("keep", "Notebook 08 manifest") if name == "bricks_manifest.csv" else ("archive", "v1 output")
            if p[2] in BRICKS and len(p) == 5 and p[3] in ("figures", "tables", "text"):
                ours = name.startswith("fig_") or re.fullmatch(r"brick_\d\d_notes\.txt", name)
                return ("keep", "Notebook 08 brick") if ours else ("archive", "v1 brick output")
            return "archive", "v1 brick folder"
        return "review", "unknown folder in outputs/"
    return "review", "unknown top-level folder"

rows = []
for f in sorted(ROOT.rglob("*")):
    if f.is_file():
        rel = f.relative_to(ROOT)
        action, reason = classify(rel)
        if action != "skip":
            rows.append({"path": str(rel), "action": action, "reason": reason, "size_mb": f.stat().st_size / 1e6})
plan = pd.DataFrame(rows)
plan.to_csv(ROOT / "tidy_plan.csv", index=False)
summary = plan.groupby("action").agg(files=("path", "size"), size_mb=("size_mb", "sum")).round(1)
print(summary, "\n")
for action in ["archive", "review"]:
    sub = plan[plan["action"] == action]
    if len(sub):
        folder = sub["path"].map(lambda x: str(Path(x).parent) + "/")
        print(f"── {action}: {len(sub)} files, by folder ──")
        print(folder.value_counts().sort_index().to_string(), "\n")
print("Full plan saved as tidy_plan.csv (open it to see every file).")

In [ ]:
# ── Apply: move the 'archive' files (only when APPLY = True) ────────────────
if not APPLY:
    print("Dry run — nothing moved. Check the plan above, then set APPLY = True and run again.")
else:
    todo = plan[plan["action"] == "archive"]
    log = []
    for rel in todo["path"]:
        src, dst = ROOT / rel, ARCHIVE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        log.append({"original_path": rel, "archived_path": str(Path(ARCHIVE_NAME) / rel)})
    index = ARCHIVE / "ARCHIVE_INDEX.csv"
    old = pd.read_csv(index) if index.exists() else pd.DataFrame()
    pd.concat([old, pd.DataFrame(log)], ignore_index=True).to_csv(index, index=False)
    for d in sorted({p for p in ROOT.rglob("*") if p.is_dir() and ARCHIVE_NAME not in p.parts and ".git" not in p.parts},
                    key=lambda p: -len(p.parts)):
        if not any(d.iterdir()):
            d.rmdir()                                              # remove folders the move left empty
    print(f"Moved {len(log)} files to {ARCHIVE_NAME}/ · index: {ARCHIVE_NAME}/ARCHIVE_INDEX.csv")

In [ ]:
# ── Undo: put every archived file back where it was (run only if you need to) ─
UNDO = False
if UNDO:
    idx = pd.read_csv(ARCHIVE / "ARCHIVE_INDEX.csv")
    for r in idx.itertuples():
        src, dst = ROOT / r.archived_path, ROOT / r.original_path
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
    print(f"Restored {len(idx)} files.")